# Rank-1 QK Experiment: $W_Q = e_2$, $W_K = e_3$

**Setup:** Same toy transformer as the base notebook — vocabulary $V=5$, context length $L=4$, embedding dimension $d = V+L = 9$ — but now the query and key projections are **rank-1** (key dimension $d_k = 1$).

**Conventions throughout:**
- Token representations are **row vectors** $x \in \mathbb{R}^{1 \times d}$.
- Matrices multiply on the **right**: $Q = XW_Q$, $K = XW_K$.
- Token $t$ at context position $p$ is embedded as $x(t,p) = e_t + e_{V+p}$, where $e_i$ is the $i$-th standard basis row vector.
- Embedding indices $0\ldots4$ are token dimensions ($t_0\ldots t_4$); indices $5\ldots8$ are position dimensions ($p_0\ldots p_3$).

## Rank-1 Projections

With $d_k = 1$, the projection matrices are **column vectors**:

$$
W_Q \in \mathbb{R}^{d \times 1}, \qquad W_K \in \mathbb{R}^{d \times 1}
$$

This gives:

$$
Q = X W_Q \in \mathbb{R}^{L \times 1}, \qquad K = X W_K \in \mathbb{R}^{L \times 1}
$$

$$
S = Q K^T \in \mathbb{R}^{L \times L}
$$

The score $S_{ij} = Q_i \cdot K_j$ is just the product of two scalars.

**Relationship matrix** (see base notebook):

$$
R = W_Q W_K^T \in \mathbb{R}^{d \times d}, \qquad S = X R X^T
$$

Because $W_Q$ and $W_K$ are rank-1 column vectors, $R$ is a **rank-1 matrix**: a single outer product $w_q w_k^T$, with at most one nonzero entry when $w_q$ and $w_k$ are basis vectors.

## Choice: $W_Q = e_2$, $W_K = e_3$

Setting $W_Q = e_2$ and $W_K = e_3$ (column vectors, 0-indexed) gives:

$$
Q_i = x_i \cdot e_2 = x_i[2], \qquad K_j = x_j \cdot e_3 = x_j[3]
$$

Index 2 is the token dimension for $t_2$, and index 3 is the token dimension for $t_3$. So:

- $Q_i = 1$ if and only if the token at context position $i$ is $t_2$, else $0$.
- $K_j = 1$ if and only if the token at context position $j$ is $t_3$, else $0$.

$$
S_{ij} = Q_i K_j = \begin{cases} 1 & \text{token at } i \text{ is } t_2 \text{ AND token at } j \text{ is } t_3 \\ 0 & \text{otherwise} \end{cases}
$$

The relationship matrix is:

$$
R = e_2^T e_3
$$

a $d \times d$ matrix with a **single 1 at position $[2, 3]$** and zeros everywhere else. It encodes a single rule: *"$t_2$ as query attends to $t_3$ as key"*.

## Attention Scale Factor $\alpha$

Standard scaled dot-product attention divides by $\sqrt{d_k}$ to prevent softmax saturation. Here we replace that with an explicit **attention scale factor** $\alpha$ that multiplies the query (but not the key):

$$
S_{ij} = (\alpha\, Q_i)\, K_j = \alpha\, Q_i K_j
$$

This is equivalent to scaling the entire score matrix: $S_{\alpha} = \alpha S$.

**Effect on softmax.** For a row with scores $(0, \alpha, 0, \ldots)$ vs $(0, 1, 0, \ldots)$, the probability on the attended position goes from $\sigma(\alpha) = e^\alpha / (e^\alpha + n-1)$ to $\sigma(1)$ as $\alpha$ becomes large. With our binary scores and $n$ visible positions, the attended weight is:

$$
P_{\text{attended}} = \frac{e^\alpha}{e^\alpha + (n-1)}
$$

For $\alpha = 1$: depends on $n$. For $\alpha = 10$: $\approx e^{10}/(e^{10}+n) \approx 1$ — near-deterministic attention.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Dimensions
V = 5           # Vocabulary size: tokens t0..t4
L = 4           # Context length
d = V + L       # Embedding dimension = 9
d_k = 1         # Key/query dimension (rank-1)

# Attention scale factor — multiplies the query before scoring.
# Larger alpha sharpens the softmax distribution.
alpha = 10

# Embedding index labels (used throughout for axis labels)
dim_labels = [f"t{i}" for i in range(V)] + [f"p{i}" for i in range(L)]

In [ ]:
def get_input_matrix(token_indices):
    """Build X in R^{L x d}. Row i = e_{t_i} + e_{V+i}."""
    tokens = torch.tensor(token_indices)
    content = F.one_hot(tokens, num_classes=V).float()          # (L, V)
    position = F.one_hot(torch.arange(L), num_classes=L).float() # (L, L)
    return torch.cat([content, position], dim=-1)               # (L, d)

# Sequence chosen so that t2 and t3 both appear at multiple positions,
# creating an interesting causal attention pattern.
# pos 0: t2  (Q=1, K=0)
# pos 1: t3  (Q=0, K=1)  <- only key position visible to pos 2 and 3
# pos 2: t2  (Q=1, K=0)
# pos 3: t3  (Q=0, K=1)
sequence = [2, 3, 2, 3]
X = get_input_matrix(sequence)
print("Sequence:", sequence)
print("X (input matrix):")
print(X.numpy())

## PyTorch Convention Note

`nn.Linear(d, d_k, bias=False)` stores its weight as shape `(d_k, d)` — transposed relative to the mathematical column vector $W_Q \in \mathbb{R}^{d \times d_k}$. The forward pass computes $XW_Q^{\text{weight},T}$, which equals $XW_Q$ in our notation.

To set $W_Q = e_2$ (column), we set `W_Q.weight[0, 2] = 1` (row vector $e_2^T$).

The mathematical column vectors recovered as:
$$
W_Q^{\text{math}} = W_Q.\text{weight}^T, \qquad W_K^{\text{math}} = W_K.\text{weight}^T
$$

So in code: `R = W_Q.weight.T @ W_K.weight` gives $R = W_Q W_K^T \in \mathbb{R}^{d \times d}$.

In [ ]:
# Rank-1 projection matrices: d -> d_k=1
W_Q = torch.nn.Linear(d, d_k, bias=False)
W_K = torch.nn.Linear(d, d_k, bias=False)

assert W_Q.weight.shape == (d_k, d) and W_K.weight.shape == (d_k, d), \
    f"Expected both weights to have shape {(d_k, d)}, got {W_Q.weight.shape} and {W_K.weight.shape}"

# Set W_Q = e_2, W_K = e_3 (0-indexed basis vectors)
with torch.no_grad():
    W_Q.weight.zero_()    # shape (1, d)
    W_Q.weight[0, 2] = 1  # selects embedding dimension 2 (token t2)
    W_K.weight.zero_()    # shape (1, d)
    W_K.weight[0, 3] = 1  # selects embedding dimension 3 (token t3)

Q = W_Q(X)  # (L, 1) — scalar query per position
K = W_K(X)  # (L, 1) — scalar key per position

assert Q.shape == (L, d_k), f"Expected Q shape {(L, d_k)}, got {tuple(Q.shape)}"
assert K.shape == (L, d_k), f"Expected  shape {(L, d_k)}, got {tuple(K.shape)}"

print("Q (query scalars per position):", Q.detach().numpy().flatten())
print("K (key scalars per position):  ", K.detach().numpy().flatten())

In [ ]:
# Relationship matrix R = W_Q W_K^T in R^{d x d}
# Using PyTorch convention: mathematical W_Q = W_Q.weight.T
R = W_Q.weight.T @ W_K.weight   # (d, 1) @ (1, d) = (d, d), rank-1

# Unscaled score matrix via S = X R X^T (equivalent to Q K^T)
S = X @ R @ X.T                 # (L, L)

# Scaled scores: alpha multiplies the query, sharpening softmax
S_scaled = alpha * S            # (L, L)

# Verify equivalence with direct computation
S_direct = (alpha * Q) @ K.T
print("Max |(alpha*Q) K^T - S_scaled|:", (S_scaled - S_direct).abs().max().item())

print("\nR (nonzero entries only):")
nz = R.nonzero()
for idx in nz:
    print(f"  R[{idx[0]}, {idx[1]}] = {R[idx[0], idx[1]].item():.1f}",
          f"  ({dim_labels[idx[0]]} -> {dim_labels[idx[1]]})")

print(f"\nS (unscaled, alpha=1):\n{S.detach().numpy()}")
print(f"\nS_scaled (alpha={alpha}):\n{S_scaled.detach().numpy()}")

In [ ]:
# Causal masking applied to the scaled scores
mask = torch.tril(torch.ones(L, L))
S_scaled_masked = S_scaled.masked_fill(mask == 0, float('-inf'))
P = F.softmax(S_scaled_masked, dim=-1)

print(f"P (attention probabilities, alpha={alpha}, causally masked):")
print(P.detach().numpy().round(4))

## Visualization

Three panels:

1. **$R$ (relationship matrix, $d \times d$):** A single nonzero entry at $[t_2, t_3]$, encoding the rule *"$t_2$-query attends to $t_3$-key"*. The dashed line separates token dimensions (left/top) from position dimensions (right/bottom).

2. **$S$ (score matrix, $L \times L$, unmasked):** Binary — entry $(i,j)=1$ iff position $i$ holds $t_2$ and position $j$ holds $t_3$.

3. **$P$ (attention probabilities, $L \times L$, causally masked):** The distribution each query position places over its past (including itself). Rows for $t_3$-positions (zero query) are uniform over the visible window.

In [ ]:
R_np = R.detach().numpy()
S_np = S.detach().numpy()
P_np = P.detach().numpy()

pos_labels = [f"pos {i}\n(t{sequence[i]})" for i in range(L)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Panel 1: R ──────────────────────────────────────────────────────────────
ax = axes[0]
R_absmax = max(np.abs(R_np).max(), 1e-6)
im = ax.imshow(R_np, cmap="RdBu_r", vmin=-R_absmax, vmax=R_absmax)
ax.set_title(r"$R = W_Q W_K^T \in \mathbb{R}^{d \times d}$" + "\n(rank-1 relationship matrix)", fontsize=11)
ax.set_xlabel("Key dimension")
ax.set_ylabel("Query dimension")
ax.set_xticks(range(d)); ax.set_xticklabels(dim_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(d)); ax.set_yticklabels(dim_labels, fontsize=8)
ax.axhline(V - 0.5, color="black", lw=1.5, ls="--")
ax.axvline(V - 0.5, color="black", lw=1.5, ls="--")
plt.colorbar(im, ax=ax, fraction=0.046)
for i in range(d):
    for j in range(d):
        if R_np[i, j] != 0:
            ax.text(j, i, f"{R_np[i,j]:.0f}", ha="center", va="center",
                    fontsize=9, fontweight="bold", color="white")

# ── Panel 2: S (unscaled, unmasked) ─────────────────────────────────────────
ax = axes[1]
im2 = ax.imshow(S_np, cmap="Blues", vmin=0, vmax=max(S_np.max(), 1))
ax.set_title(r"$S = X R X^T \in \mathbb{R}^{L \times L}$" + "\n(scores, unscaled, unmasked)", fontsize=11)
ax.set_xlabel("Key position j")
ax.set_ylabel("Query position i")
ax.set_xticks(range(L)); ax.set_xticklabels(pos_labels, fontsize=9)
ax.set_yticks(range(L)); ax.set_yticklabels(pos_labels, fontsize=9)
plt.colorbar(im2, ax=ax, fraction=0.046)
for i in range(L):
    for j in range(L):
        ax.text(j, i, f"{S_np[i,j]:.0f}", ha="center", va="center", fontsize=12,
                color="white" if S_np[i, j] > 0.5 else "black")

# ── Panel 3: P (scaled, causally masked) ─────────────────────────────────────
ax = axes[2]
im3 = ax.imshow(P_np, cmap="Oranges", vmin=0, vmax=1)
ax.set_title(
    r"$P = \mathrm{softmax}(\alpha S_{\mathrm{masked}})$" + f"\n(attention probs, $\\alpha={alpha}$, causal)",
    fontsize=11
)
ax.set_xlabel("Key position j")
ax.set_ylabel("Query position i")
ax.set_xticks(range(L)); ax.set_xticklabels(pos_labels, fontsize=9)
ax.set_yticks(range(L)); ax.set_yticklabels(pos_labels, fontsize=9)
plt.colorbar(im3, ax=ax, fraction=0.046)
for i in range(L):
    for j in range(L):
        ax.text(j, i, f"{P_np[i,j]:.2f}", ha="center", va="center", fontsize=9,
                color="white" if P_np[i, j] > 0.5 else "black")

plt.suptitle(
    r"Rank-1 attention: $W_Q = e_2$ (queries $t_2$), $W_K = e_3$ (keys $t_3$)"
    f",  $\\alpha={alpha}$\nSequence: {sequence}",
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()

## Interpretation

**$R$ is a single-rule lookup table.** With $W_Q = e_2$ and $W_K = e_3$, the relationship matrix has exactly one nonzero entry $R[2, 3] = 1$. This encodes a single directional rule operating purely in token space:

> *token $t_2$ (as query) attends to token $t_3$ (as key)*

**$S$ is binary.** Because $x(t,p)$ is a 0/1 vector, each query scalar $Q_i = x_i[2] \in \{0,1\}$ and each key scalar $K_j = x_j[3] \in \{0,1\}$, so $S_{ij} = Q_i K_j \in \{0,1\}$.

**Attention concentrates on $t_3$-positions.** After causal masking:
- $t_2$-query positions (pos 0 and pos 2) have $Q_i = 1$: they attend only to visible $t_3$-positions.
- $t_3$-query positions (pos 1 and pos 3) have $Q_i = 0$: all their scores are 0, so softmax gives a **uniform distribution** over the visible window — these positions are "blind" under this QK pair.

**No position information is used.** $R$ has no nonzero entries in the position block (indices $V\ldots V+L-1$). Attention here is determined **entirely by token identity**, not by where tokens appear in the context. The next experiment will introduce position basis vectors into $W_Q$ or $W_K$ to break this symmetry.

## Value and Output Matrices

To produce a next-token prediction we need two more matrices.

### Value matrix $W_V \in \mathbb{R}^{d \times d_v}$

Applied to the input to form the **value vectors**:

$$
V_{\text{mat}} = X W_V \in \mathbb{R}^{L \times d_v}
$$

The attention output is then:

$$
H = P V_{\text{mat}} \in \mathbb{R}^{L \times d_v}
$$

Row $i$ of $H$ is a weighted sum of value vectors: $h_i = \sum_j P_{ij} v_j$.

We use $W_V = I_d$ (the $d \times d$ identity), so $V_{\text{mat}} = X$ — each value vector is just the raw token embedding. This means $H = PX$: each row $h_i$ is an **attention-weighted average of the input embeddings**.

### Output matrix $W_O \in \mathbb{R}^{d_v \times V}$

Projects the attention output down to a logit vector over the vocabulary:

$$
z = h_L \, W_O \in \mathbb{R}^{1 \times V}
$$

$$
\hat{p} = \text{softmax}(z) \in \Delta^{V-1}
$$

We use $W_O = I_d[{:V}, :]^T \in \mathbb{R}^{d \times V}$ — i.e. the first $V$ columns of the identity — which simply **extracts the token block** (dimensions $0\ldots V{-}1$) of $h_L$ and discards the position block (dimensions $V\ldots d{-}1$).

### Why this gives a frequency-weighted combination

Because each embedding $x(t,p) = e_t + e_{V+p}$ has a 1 in dimension $t$ and a 1 in a position dimension, the token block of $h_L$ is:

$$
h_L[:V] = \sum_j P_{Lj} \, e_{t_j}
$$

This is a **convex combination of one-hot token vectors**, weighted by how much position $L$ attends to each previous position $j$. It is a soft histogram over the attended tokens — a frequency count, re-weighted by attention.

In [ ]:
# W_V = I_d: values are the raw embeddings, so V_mat = X
W_V = torch.eye(d)                      # (d, d)
V_mat = X @ W_V                         # (L, d) == X

# H = P V_mat: attention-weighted average of embeddings
H = P @ V_mat                           # (L, d)

# W_O extracts the token block (first V dims), discarding position dims
W_O = torch.eye(d)[:, :V]              # (d, V)

# Logits and prediction for the last position (the "next token" query)
h_last = H[-1]                          # (d,)
z = h_last @ W_O                        # (V,) — token-block of h_last
p_next = F.softmax(z, dim=-1)           # (V,) — distribution over vocabulary

print("h_last (attention output for final position):")
print(h_last.detach().numpy().round(4))
print(f"\nToken block of h_last (= z, pre-softmax logits):")
print(z.detach().numpy().round(4))
print(f"\np_next (predicted next-token distribution):")
for tok_id, prob in enumerate(p_next.detach().numpy()):
    bar = "█" * int(prob * 40)
    print(f"  t{tok_id}: {prob:.4f}  {bar}")
print(f"\nMost likely next token: t{p_next.argmax().item()}  (prob={p_next.max().item():.4f})")

## Interpretation: Prediction for This Example

The last position (pos 3) holds token $t_3$, which has **query scalar $Q_3 = 0$** — it is a key token, not a query token under this QK pair. All its scores are 0, so after causal masking the softmax is uniform over the four visible positions:

$$
P[3, :] = [0.25,\ 0.25,\ 0.25,\ 0.25]
$$

The attention output is therefore the **uniform average of all four input embeddings**:

$$
h_{\text{last}} = \tfrac{1}{4}(x_0 + x_1 + x_2 + x_3)
$$

The sequence is $[t_2, t_3, t_2, t_3]$, so the token block is:

$$
h_{\text{last}}[:V] = \tfrac{1}{4}(e_2 + e_3 + e_2 + e_3) = \tfrac{1}{2} e_2 + \tfrac{1}{2} e_3
$$

After softmax, $t_2$ and $t_3$ each receive equal probability $\approx 0.5$ (with small residual on $t_0\ldots t_4$ from the position dimensions leaking through, which are symmetric and cancel after softmax).

**Contrast with a $t_2$-query position** (e.g. pos 2, $\alpha=10$): its attention is near-deterministic on the most recent $t_3$ it has seen (pos 1), so $h[2] \approx x_1 = e_3 + e_6$ and the prediction strongly favours $t_3$. The $\alpha$ parameter controls how sharply the model commits to the attended token.

---

## Example 2: Implementing a Conditional Rule with Rank-1 Values

**Differences from Example 1:**

| | Example 1 | Example 2 |
|---|---|---|
| Sequence | [t2, t3, t2, t3] | [t0, t1, t0, t3] |
| $\alpha$ | 10 | 5 |
| Masking | causal | **none** |
| $d_v$ | $d$ (identity $W_V$) | **1** (rank-1 $W_V$) |

**Target rule:** *"If the final token is $t_3$ and $t_1$ appears anywhere in the context, predict $t_2$. Otherwise predict $t_4$."*

### Design of $W_Q$ and $W_K$

The final position holds $t_3$. We want it to query for $t_1$ anywhere in the (unmasked) context:

$$
W_Q = e_3, \quad W_K = e_1 \implies Q_i = x_i[3],\quad K_j = x_j[1]
$$

- $Q_i = 1$ iff token at position $i$ is $t_3$ (only pos 3 fires).
- $K_j = 1$ iff token at position $j$ is $t_1$ (only pos 1 fires).
- $S_{ij} = \alpha Q_i K_j$ is nonzero only at $(i,j) = (3,1)$.

Without masking, position 3 can attend to the $t_1$ at position 1 regardless of where it appears.

### Design of $W_V$ (rank-1, $d_v = 1$)

$$
W_V = e_1 \in \mathbb{R}^{d \times 1} \implies v_j = x_j[1] = \begin{cases} 1 & \text{token at } j \text{ is } t_1 \\ 0 & \text{otherwise} \end{cases}
$$

The attention output for the last position is then:

$$
h_{\text{last}} = \sum_j P[3,j]\, v_j = P[3,1] \approx 1 \quad \text{(when } t_1 \text{ is present)}
$$

$h_{\text{last}}$ is a scalar in $[0,1]$: it reads out *how strongly the last position attended to a $t_1$ token*.

### Design of $W_O$ and the "otherwise" limitation

$W_O \in \mathbb{R}^{d_v \times V} = \mathbb{R}^{1 \times V}$ maps the scalar $h_{\text{last}}$ to logits over $V$ tokens:

$$
z = h_{\text{last}} \cdot w_O^\top \in \mathbb{R}^V
$$

To predict $t_2$ when $h_{\text{last}} \approx 1$: set $W_O = \gamma\, e_2^\top$ (large positive weight on $t_2$).

**The "otherwise" problem.** When no $t_1$ is present, $h_{\text{last}} \approx 0$, so $z \approx \mathbf{0}$ regardless of $W_O$ — softmax returns a **uniform distribution**, not a prediction of $t_4$. A bias term or a second attention head would be needed to break this symmetry. This is a fundamental limitation of rank-1, bias-free attention for implementing "default" behaviour.

In [ ]:
# ── Example 2 parameters ─────────────────────────────────────────────────────
alpha2 = 5    # attention scale factor
gamma  = 5    # output projection scale (W_O strength)

sequence2 = [0, 1, 0, 3]   # t0, t1, t0, t3
X2 = get_input_matrix(sequence2)

print("Sequence:", sequence2)
print("X2 (input matrix):\n", X2.numpy())

# ── QK projections: W_Q = e_3, W_K = e_1 ────────────────────────────────────
W_Q2 = torch.nn.Linear(d, d_k, bias=False)
W_K2 = torch.nn.Linear(d, d_k, bias=False)

with torch.no_grad():
    W_Q2.weight.zero_(); W_Q2.weight[0, 3] = 1   # queries t3 positions
    W_K2.weight.zero_(); W_K2.weight[0, 1] = 1   # keys t1 positions

Q2 = W_Q2(X2)   # (L, 1)
K2 = W_K2(X2)   # (L, 1)

print("\nQ2 (query scalars):", Q2.detach().numpy().flatten())
print("K2 (key scalars):  ", K2.detach().numpy().flatten())

# ── Relationship matrix and scores (no masking) ───────────────────────────────
R2      = W_Q2.weight.T @ W_K2.weight   # (d, d), rank-1, 1 at [3, 1]
S2      = X2 @ R2 @ X2.T               # (L, L)
S2_scaled = alpha2 * S2                 # (L, L)

print("\nR2 nonzero entries:")
for idx in R2.nonzero():
    print(f"  R2[{idx[0]}, {idx[1]}] = {R2[idx[0],idx[1]].item():.1f}",
          f"  ({dim_labels[idx[0]]} -> {dim_labels[idx[1]]})")

print("\nS2 (unscaled):\n", S2.detach().numpy())
print(f"\nS2_scaled (alpha={alpha2}):\n", S2_scaled.detach().numpy())

# ── Attention probabilities (no masking) ─────────────────────────────────────
P2 = F.softmax(S2_scaled, dim=-1)
print(f"\nP2 (no masking, alpha={alpha2}):\n", P2.detach().numpy().round(4))

In [ ]:
# ── Rank-1 value matrix: W_V = e_1, d_v = 1 ──────────────────────────────────
# v_j = X2[j, 1] = 1 iff token at position j is t1, else 0
W_V2 = torch.zeros(d, 1)
W_V2[1, 0] = 1.0             # selects embedding dimension 1 (token t1)

V_mat2 = X2 @ W_V2           # (L, 1) — scalar value per position
H2     = P2 @ V_mat2          # (L, 1) — scalar attention output per position

print("Value scalars V_mat2 (1 = t1 token):", V_mat2.detach().numpy().flatten())
print("Attention outputs H2:", H2.detach().numpy().flatten().round(4))
print(f"\nh_last = H2[-1] = {H2[-1].item():.4f}  (≈1 when t1 is strongly attended)")

# ── Output matrix: W_O = gamma * e_2^T, shape (1, V) ─────────────────────────
# z = h_last * W_O  →  logit on t2 proportional to h_last
W_O2 = torch.zeros(1, V)
W_O2[0, 2] = gamma           # only t2 gets a nonzero logit

h_last2 = H2[-1].unsqueeze(0)          # (1, 1)
z2      = h_last2 @ W_O2               # (1, V)
p_next2 = F.softmax(z2, dim=-1).squeeze()  # (V,)

print(f"\nW_O2 (output projection): {W_O2.numpy()}")
print(f"z2 (logits): {z2.detach().numpy().round(4)}")
print(f"\np_next2 (predicted next-token distribution):")
for tok_id, prob in enumerate(p_next2.detach().numpy()):
    bar = "█" * int(prob * 40)
    print(f"  t{tok_id}: {prob:.4f}  {bar}")
print(f"\nMost likely next token: t{p_next2.argmax().item()}  (prob={p_next2.max().item():.4f})")

In [ ]:
R2_np  = R2.detach().numpy()
S2_np  = S2.detach().numpy()
P2_np  = P2.detach().numpy()
p_next2_np = p_next2.detach().numpy()

pos_labels2 = [f"pos {i}\n(t{sequence2[i]})" for i in range(L)]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# ── Panel 1: R2 ───────────────────────────────────────────────────────────────
ax = axes[0]
R2_absmax = max(np.abs(R2_np).max(), 1e-6)
im = ax.imshow(R2_np, cmap="RdBu_r", vmin=-R2_absmax, vmax=R2_absmax)
ax.set_title(r"$R_2 = W_Q W_K^T$" + "\n" + r"($t_3$ queries $t_1$)", fontsize=11)
ax.set_xlabel("Key dimension"); ax.set_ylabel("Query dimension")
ax.set_xticks(range(d)); ax.set_xticklabels(dim_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(d)); ax.set_yticklabels(dim_labels, fontsize=8)
ax.axhline(V - 0.5, color="black", lw=1.5, ls="--")
ax.axvline(V - 0.5, color="black", lw=1.5, ls="--")
plt.colorbar(im, ax=ax, fraction=0.046)
for i in range(d):
    for j in range(d):
        if R2_np[i, j] != 0:
            ax.text(j, i, f"{R2_np[i,j]:.0f}", ha="center", va="center",
                    fontsize=9, fontweight="bold", color="white")

# ── Panel 2: S2 (unscaled, no masking) ───────────────────────────────────────
ax = axes[1]
im2 = ax.imshow(S2_np, cmap="Blues", vmin=0, vmax=max(S2_np.max(), 1))
ax.set_title(r"$S_2 = X R_2 X^T$" + "\n(scores, unscaled, no mask)", fontsize=11)
ax.set_xlabel("Key position j"); ax.set_ylabel("Query position i")
ax.set_xticks(range(L)); ax.set_xticklabels(pos_labels2, fontsize=9)
ax.set_yticks(range(L)); ax.set_yticklabels(pos_labels2, fontsize=9)
plt.colorbar(im2, ax=ax, fraction=0.046)
for i in range(L):
    for j in range(L):
        ax.text(j, i, f"{S2_np[i,j]:.0f}", ha="center", va="center", fontsize=12,
                color="white" if S2_np[i, j] > 0.5 else "black")

# ── Panel 3: P2 (no masking) ─────────────────────────────────────────────────
ax = axes[2]
im3 = ax.imshow(P2_np, cmap="Oranges", vmin=0, vmax=1)
ax.set_title(r"$P_2 = \mathrm{softmax}(\alpha_2 S_2)$" + f"\n(no mask, $\\alpha={alpha2}$)", fontsize=11)
ax.set_xlabel("Key position j"); ax.set_ylabel("Query position i")
ax.set_xticks(range(L)); ax.set_xticklabels(pos_labels2, fontsize=9)
ax.set_yticks(range(L)); ax.set_yticklabels(pos_labels2, fontsize=9)
plt.colorbar(im3, ax=ax, fraction=0.046)
for i in range(L):
    for j in range(L):
        ax.text(j, i, f"{P2_np[i,j]:.2f}", ha="center", va="center", fontsize=9,
                color="white" if P2_np[i, j] > 0.5 else "black")

# ── Panel 4: next-token prediction ───────────────────────────────────────────
ax = axes[3]
colors = ["steelblue"] * V
colors[2] = "darkorange"   # highlight t2 (the predicted token)
ax.bar(range(V), p_next2_np, color=colors)
ax.set_title(f"$\\hat{{p}}$ (next-token prediction)\n"
             f"$h_{{\\mathrm{{last}}}}={H2[-1].item():.3f}$,  $\\gamma={gamma}$", fontsize=11)
ax.set_xlabel("Token"); ax.set_ylabel("Probability")
ax.set_xticks(range(V)); ax.set_xticklabels([f"t{i}" for i in range(V)])
ax.set_ylim(0, 1)
for i, p in enumerate(p_next2_np):
    ax.text(i, p + 0.02, f"{p:.3f}", ha="center", va="bottom", fontsize=9)

plt.suptitle(
    r"Example 2: $W_Q=e_3$, $W_K=e_1$, $W_V=e_1$ (rank-1), no masking"
    f"\nSequence: {sequence2},  $\\alpha={alpha2}$,  $\\gamma={gamma}$",
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()

## Interpretation: Example 2

**What worked — the "if t3 and t1" branch.**

The single nonzero in $R_2$ at $[t_3, t_1]$ creates exactly one nonzero score: $S_2[3,1] = 1$ (pos 3 is $t_3$, pos 1 is $t_1$). After scaling by $\alpha=5$:

$$
P_2[3,:] = \text{softmax}([0,\ 5,\ 0,\ 0]) \approx [0.007,\ 0.980,\ 0.007,\ 0.007]
$$

The last position attends almost entirely to position 1 ($t_1$). Since $v_1 = 1$ (token is $t_1$) and all other $v_j = 0$:

$$
h_{\text{last}} = P_2[3,1] \approx 0.980
$$

With $W_O = \gamma\, e_2^\top$ and $\gamma=5$:

$$
z = [0,\ 0,\ 0.980 \times 5,\ 0,\ 0] = [0,\ 0,\ 4.9,\ 0,\ 0]
$$

$$
\hat{p}(t_2) = \frac{e^{4.9}}{e^{4.9} + 4} \approx 0.97
$$

The rule *"if final token is $t_3$ and $t_1$ is present, predict $t_2$"* is implemented successfully.

**What failed — the "otherwise predict $t_4$" branch.**

When no $t_1$ is present in the context, all value scalars $v_j = 0$, so $h_{\text{last}} = 0$ and $z = \mathbf{0}$. Softmax of the zero vector is uniform over all $V$ tokens — the model is equally confused about all of them. No choice of $W_O$ can fix this: if $h_{\text{last}} = 0$, no signal reaches $W_O$.

**What would be needed for the "otherwise" branch:**

| Approach | Idea |
|---|---|
| Bias in $W_O$ | $z = h_{\text{last}} \cdot w_O^\top + b$ — $b$ provides a default logit, e.g. large $b[t_4]$ |
| Second attention head | Head 1 detects $t_1$; head 2 provides a constant background signal on $t_4$ |
| Non-zero $W_V$ on $t_4$ | Make values nonzero at non-$t_1$ positions, so $h_{\text{last}} < 1$ signals their presence |

The last option is particularly interesting: if $W_V = e_1 - c\, e_4$ for some $c > 0$, then non-$t_1$ positions contribute negative signal, and $W_O$ can be designed to flip from $t_2$ to $t_4$ as $h_{\text{last}}$ decreases. But this requires $d_v \geq 1$ with signed weights — still achievable at rank-1.

---

## Example 3: Implementing Both Branches with $d_v = 2$

Everything is the same as Example 2 ($W_Q = e_3$, $W_K = e_1$, $\alpha=5$, no masking) except we now use a **rank-2 value matrix** ($d_v = 2$) that gives the model two independent scalar channels per position.

### The two-channel value design

$$
W_V = \begin{bmatrix} w_0 & w_1 \end{bmatrix} \in \mathbb{R}^{d \times 2}
$$

**Channel 0** — detects $t_1$:

$$
w_0 = e_1 \implies v_j^{(0)} = x_j[1] = \begin{cases} 1 & \text{token at } j \text{ is } t_1 \\ 0 & \text{otherwise} \end{cases}
$$

**Channel 1** — constant background:

$$
w_1 = \tfrac{1}{2}\mathbf{1}_d \implies v_j^{(1)} = x_j \cdot \tfrac{1}{2}\mathbf{1}_d = \tfrac{1}{2}(1 + 1) = 1 \quad \forall j
$$

The second equality holds because every token embedding $x(t,p) = e_t + e_{V+p}$ has exactly **two** non-zero entries, both equal to 1. So the dot product with $\frac{1}{2}\mathbf{1}$ always equals $1$, regardless of the token or position.

Since $P$ is a stochastic matrix (rows sum to 1), channel 1 of the attention output is always exactly 1:

$$
h^{(1)} = \sum_j P_{ij} \cdot 1 = 1 \quad \forall i
$$

This gives a **free bias** — a constant signal available to $W_O$ even when the attention pattern carries no information about $t_1$.

### Output matrix design

$W_O \in \mathbb{R}^{2 \times V}$ routes the two channels to vocabulary logits:

$$
z = h_{\text{last}} \, W_O = h^{(0)} \cdot W_O[0,:] + \underbrace{1}_{\displaystyle h^{(1)}} \cdot W_O[1,:]
$$

We set:

$$
W_O[0,:] = \gamma\,(e_2 - e_4)^\top \qquad \text{(channel 0: boost } t_2\text{, suppress }t_4\text{)}
$$

$$
W_O[1,:] = \gamma\, e_4^\top \qquad\qquad\quad \text{(channel 1: default boost for }t_4\text{)}
$$

**When $t_1$ is present** ($h^{(0)} \approx 1$):

$$
z \approx \gamma(e_2 - e_4) + \gamma e_4 = \gamma\, e_2 \implies \hat{p}(t_2) \approx 1 \checkmark
$$

**When $t_1$ is absent** ($h^{(0)} \approx 0$):

$$
z \approx \gamma\, e_4 \implies \hat{p}(t_4) \approx 1 \checkmark
$$

In [ ]:
gamma3 = 5   # output scale (same alpha2=5 and W_Q2/W_K2 reused from Example 2)
d_v3   = 2

# ── Value matrix W_V3: d x 2 ──────────────────────────────────────────────────
W_V3 = torch.zeros(d, d_v3)
W_V3[:, 0] = torch.zeros(d); W_V3[1, 0] = 1.0   # channel 0: e_1 (detects t1)
W_V3[:, 1] = 0.5 * torch.ones(d)                 # channel 1: 1/2 * 1  (constant = 1 per token)

# ── Output matrix W_O3: 2 x V ────────────────────────────────────────────────
W_O3 = torch.zeros(d_v3, V)
W_O3[0, 2] =  gamma3   # channel 0 boosts t2 ...
W_O3[0, 4] = -gamma3   # ... and suppresses t4
W_O3[1, 4] =  gamma3   # channel 1 (default) boosts t4

print("W_V3 (d × 2):")
print(W_V3.numpy())
print("\nW_O3 (2 × V):")
print(W_O3.numpy())
print(f"\nExpected logits when t1 present  (h=[1,1]):  z = {(W_O3[0]+W_O3[1]).numpy()}")
print(f"Expected logits when t1 absent   (h=[0,1]):  z = {W_O3[1].numpy()}")

In [ ]:
def run_example3(seq, label):
    """Full forward pass for Example 3 given a token sequence."""
    X_     = get_input_matrix(seq)

    # QK (same as Example 2: W_Q=e_3, W_K=e_1)
    Q_     = W_Q2(X_)
    K_     = W_K2(X_)
    S_     = alpha2 * (Q_ @ K_.T)          # scaled scores, no masking
    P_     = F.softmax(S_, dim=-1)

    # Value and output
    V_mat_ = X_ @ W_V3                     # (L, 2)
    H_     = P_ @ V_mat_                    # (L, 2)
    h_last_= H_[-1]                         # (2,)
    z_     = h_last_.unsqueeze(0) @ W_O3    # (1, V)
    p_next_= F.softmax(z_, dim=-1).squeeze()

    print(f"\n{'='*55}")
    print(f"Sequence: {seq}  [{label}]")
    print(f"  Q  (scalars): {Q_.detach().numpy().flatten()}")
    print(f"  K  (scalars): {K_.detach().numpy().flatten()}")
    print(f"  P[-1] (last row): {P_[-1].detach().numpy().round(4)}")
    print(f"  V_mat (channel 0, 1): {V_mat_.detach().numpy().T}")
    print(f"  h_last = [{h_last_[0].item():.4f}, {h_last_[1].item():.4f}]")
    print(f"  z (logits):  {z_.detach().numpy().round(4)}")
    print(f"  p_next: {[f't{i}={p:.3f}' for i, p in enumerate(p_next_.detach().numpy())]}")
    print(f"  → Predicted: t{p_next_.argmax().item()}  (p={p_next_.max().item():.4f})")
    return P_.detach().numpy(), p_next_.detach().numpy(), h_last_.detach().numpy()

# Branch 1: t1 IS present — expect t2
P3a, pn3a, h3a = run_example3([0, 1, 0, 3], "t1 present → expect t2")

# Branch 2: t1 is ABSENT — expect t4
P3b, pn3b, h3b = run_example3([0, 0, 0, 3], "t1 absent  → expect t4")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

cases = [
    ([0,1,0,3], "t1 present → predict $t_2$", P3a, pn3a, h3a, 2),
    ([0,0,0,3], "t1 absent  → predict $t_4$", P3b, pn3b, h3b, 4),
]

for row, (seq, title, P_, pn, h_last, expected) in enumerate(cases):
    pos_labels_ = [f"pos {i}\n(t{seq[i]})" for i in range(L)]

    # ── Panel A: Attention matrix P ───────────────────────────────────────────
    ax = axes[row, 0]
    im = ax.imshow(P_, cmap="Oranges", vmin=0, vmax=1)
    ax.set_title(f"$P$ — {title}", fontsize=10)
    ax.set_xlabel("Key pos"); ax.set_ylabel("Query pos")
    ax.set_xticks(range(L)); ax.set_xticklabels(pos_labels_, fontsize=8)
    ax.set_yticks(range(L)); ax.set_yticklabels(pos_labels_, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
    for i in range(L):
        for j in range(L):
            ax.text(j, i, f"{P_[i,j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if P_[i,j] > 0.5 else "black")

    # ── Panel B: h_last as a 2-channel bar ───────────────────────────────────
    ax = axes[row, 1]
    ch_colors = ["steelblue", "slategray"]
    bars = ax.bar(["ch 0\n(detects $t_1$)", "ch 1\n(constant)"], h_last,
                  color=ch_colors, width=0.5)
    ax.set_ylim(0, 1.2)
    ax.set_title(f"$h_{{\\mathrm{{last}}}}$ (2-channel attention output)", fontsize=10)
    ax.set_ylabel("Value")
    for bar, val in zip(bars, h_last):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.03, f"{val:.3f}",
                ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.axhline(1.0, color="gray", lw=1, ls="--", alpha=0.5)

    # ── Panel C: next-token prediction ───────────────────────────────────────
    ax = axes[row, 2]
    colors = ["steelblue"] * V
    colors[expected] = "darkorange"
    ax.bar(range(V), pn, color=colors)
    ax.set_ylim(0, 1.1)
    ax.set_title(f"$\\hat{{p}}$ — predicted next token", fontsize=10)
    ax.set_xlabel("Token"); ax.set_ylabel("Probability")
    ax.set_xticks(range(V)); ax.set_xticklabels([f"t{i}" for i in range(V)])
    for i, p in enumerate(pn):
        ax.text(i, p + 0.02, f"{p:.3f}", ha="center", va="bottom", fontsize=9)

plt.suptitle(
    r"Example 3: $d_v=2$, $W_V = [e_1\ |\ \frac{1}{2}\mathbf{1}]$, "
    r"$W_Q=e_3$, $W_K=e_1$, no masking, $\alpha=5$, $\gamma=5$"
    "\nTop: $t_1$ present (sequence [0,1,0,3])  —  Bottom: $t_1$ absent (sequence [0,0,0,3])",
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()

## Interpretation: Example 3

**Channel 1 as a free bias.** The key insight is that $v_j^{(1)} = 1$ for every token in this embedding scheme, so $h^{(1)} = \sum_j P_{ij} \cdot 1 = 1$ exactly, for any attention pattern and any input sequence. Channel 1 is not a function of the context — it is a constant that $W_O$ can treat as a bias term. This turns the output equation into:

$$
z = h^{(0)} \cdot W_O[0,:] + W_O[1,:]
$$

where $W_O[1,:]$ is effectively a **fixed bias logit vector** and $h^{(0)}$ modulates an additive correction.

**The two branches:**

- **$t_1$ present:** $h^{(0)} \approx 0.98$, so $z \approx \gamma(e_2 - e_4) + \gamma e_4 = \gamma e_2$. The $-e_4$ from channel 0 exactly cancels the $+e_4$ from the bias, leaving only $t_2$.

- **$t_1$ absent:** $h^{(0)} \approx 0$ (uniform attention, no $t_1$ values), so $z \approx \gamma e_4$. The default bias wins and $t_4$ is predicted.

**Why the cancellation is exact.** This is the key algebraic trick:

$$
W_O[0,:] + W_O[1,:] = \gamma(e_2 - e_4) + \gamma e_4 = \gamma e_2
$$

The $e_4$ components cancel by design. This is only possible because channel 1 provides a known, constant value of 1 — if it were data-dependent, the cancellation would only be approximate.

**Connection to the residual stream.** In real transformers, the MLP and attention outputs are added to a persistent residual stream, which plays the same role as channel 1 here: it provides a "prior" that subsequent layers can selectively override. The $d_v=2$ construction here is a minimal analogue of this architecture.

---

## Example 4: Position-Specific Rule with $d_v = 2$

This experiment keeps the **same two-channel value/output construction** as Example 3, but changes the **key projection** so the final query reads from a specific context position rather than searching for a token anywhere in the sequence.

**Target rule (within the same fixed-final-token setup as Example 3):**

*If the final token is $t_3$ and the token at position 2 (second from the end) is $t_1$, predict $t_2$. Otherwise predict $t_4$.*

### What changes from Example 3

- $W_Q = e_3$ is unchanged: only $t_3$ positions issue a nonzero query.
- $W_K = e_{V+2}$ is now **position-specific**: only position 2 has a nonzero key.
- $W_V$ and $W_O$ are unchanged from Example 3:

$$
W_V = \begin{bmatrix} e_1 & \tfrac{1}{2}\mathbf{1} \end{bmatrix},
\qquad
W_O[0,:] = \gamma(e_2 - e_4)^\top,
\qquad
W_O[1,:] = \gamma e_4^\top
$$

### Why this works

For any sequence whose final token is $t_3$, the last-row score vector is approximately

$$
S_{\mathrm{last}, :} = \alpha [0,\ 0,\ 1,\ 0],
$$

because only position 2 has key scalar 1. So the final position attends almost entirely to **position 2**. Channel 0 of the value matrix then reads whether that attended token is $t_1$:

$$
h_{\mathrm{last}}^{(0)} \approx \begin{cases} 1 & \text{if token at position 2 is } t_1 \\ 0 & \text{otherwise} \end{cases}
$$

Channel 1 remains the constant background signal $h_{\mathrm{last}}^{(1)} = 1$, so the same output algebra as Example 3 implements the two branches:

- if position 2 contains $t_1$, output $t_2$
- otherwise, output $t_4$

The important change is that the branch now depends on **where** the token appears, not just whether it appears anywhere in the context.

In [ ]:
alpha4 = 5
gamma4 = 5
d_v4 = 2

# QK: query on t3, key on position 2 (second from the end).
W_Q4 = torch.nn.Linear(d, d_k, bias=False)
W_K4 = torch.nn.Linear(d, d_k, bias=False)

with torch.no_grad():
    W_Q4.weight.zero_()
    W_Q4.weight[0, 3] = 1          # query fires on token t3
    W_K4.weight.zero_()
    W_K4.weight[0, V + 2] = 1      # key fires only at position 2

# Same two-channel value/output construction as Example 3.
W_V4 = torch.zeros(d, d_v4)
W_V4[1, 0] = 1.0                   # channel 0: detects token t1
W_V4[:, 1] = 0.5 * torch.ones(d)   # channel 1: constant background = 1 per token

W_O4 = torch.zeros(d_v4, V)
W_O4[0, 2] = gamma4
W_O4[0, 4] = -gamma4
W_O4[1, 4] = gamma4

def run_example4(seq, label):
    """Full forward pass for the position-specific rule in Example 4."""
    X_ = get_input_matrix(seq)

    Q_ = W_Q4(X_)
    K_ = W_K4(X_)
    S_ = alpha4 * (Q_ @ K_.T)
    P_ = F.softmax(S_, dim=-1)

    V_mat_ = X_ @ W_V4
    H_ = P_ @ V_mat_
    h_last_ = H_[-1]
    z_ = h_last_.unsqueeze(0) @ W_O4
    p_next_ = F.softmax(z_, dim=-1).squeeze()

    print(f"\n{'=' * 60}")
    print(f"Sequence: {seq}  [{label}]")
    print(f"  K (position-2 detector): {K_.detach().numpy().flatten()}")
    print(f"  P[-1] (last row):        {P_[-1].detach().numpy().round(4)}")
    print(f"  V_mat (channel 0, 1):    {V_mat_.detach().numpy().T}")
    print(f"  h_last = [{h_last_[0].item():.4f}, {h_last_[1].item():.4f}]")
    print(f"  z (logits):              {z_.detach().numpy().round(4)}")
    print(f"  p_next: {[f't{i}={p:.3f}' for i, p in enumerate(p_next_.detach().numpy())]}")
    print(f"  -> Predicted: t{p_next_.argmax().item()}  (p={p_next_.max().item():.4f})")
    return P_.detach().numpy(), p_next_.detach().numpy(), h_last_.detach().numpy()

# Branch 1: position 2 contains t1 -> expect t2
P4a, pn4a, h4a = run_example4([0, 0, 1, 3], "pos 2 = t1 -> expect t2")

# Branch 2: t1 may appear elsewhere, but position 2 does not contain t1 -> expect t4
P4b, pn4b, h4b = run_example4([0, 1, 0, 3], "pos 2 != t1 -> expect t4")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

cases4 = [
    ([0, 0, 1, 3], "position 2 holds $t_1$ -> predict $t_2$", P4a, pn4a, h4a, 2),
    ([0, 1, 0, 3], "position 2 does not hold $t_1$ -> predict $t_4$", P4b, pn4b, h4b, 4),
]

for row, (seq, title, P_, pn, h_last, expected) in enumerate(cases4):
    pos_labels_ = [f"pos {i}\\n(t{seq[i]})" for i in range(L)]

    ax = axes[row, 0]
    im = ax.imshow(P_, cmap="Oranges", vmin=0, vmax=1)
    ax.set_title(f"$P$ — {title}", fontsize=10)
    ax.set_xlabel("Key pos")
    ax.set_ylabel("Query pos")
    ax.set_xticks(range(L)); ax.set_xticklabels(pos_labels_, fontsize=8)
    ax.set_yticks(range(L)); ax.set_yticklabels(pos_labels_, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
    for i in range(L):
        for j in range(L):
            ax.text(j, i, f"{P_[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if P_[i, j] > 0.5 else "black")

    ax = axes[row, 1]
    ch_colors = ["steelblue", "slategray"]
    bars = ax.bar(["ch 0\\n(token=$t_1$)", "ch 1\\n(constant)"], h_last, color=ch_colors, width=0.5)
    ax.set_ylim(0, 1.2)
    ax.set_title(r"$h_{\mathrm{last}}$ (2-channel attention output)", fontsize=10)
    ax.set_ylabel("Value")
    for bar, val in zip(bars, h_last):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.03, f"{val:.3f}",
                ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.axhline(1.0, color="gray", lw=1, ls="--", alpha=0.5)

    ax = axes[row, 2]
    colors = ["steelblue"] * V
    colors[expected] = "darkorange"
    ax.bar(range(V), pn, color=colors)
    ax.set_ylim(0, 1.1)
    ax.set_title(r"$\hat{p}$ — predicted next token", fontsize=10)
    ax.set_xlabel("Token")
    ax.set_ylabel("Probability")
    ax.set_xticks(range(V)); ax.set_xticklabels([f"t{i}" for i in range(V)])
    for i, p in enumerate(pn):
        ax.text(i, p + 0.02, f"{p:.3f}", ha="center", va="bottom", fontsize=9)

plt.suptitle(
    r"Example 4: position-specific rule with $W_Q=e_3$, $W_K=e_{V+2}$, $d_v=2$"
    f"\nTop: sequence [0, 0, 1, 3]  —  Bottom: sequence [0, 1, 0, 3],  $\\alpha={alpha4}$,  $\\gamma={gamma4}$",
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()

## Interpretation: Example 4

**The head now reads a fixed slot in the context.** In Example 3 the key projection looked for token $t_1$ anywhere. Here the key projection is purely positional: $W_K = e_{V+2}$, so the last position sends almost all of its attention to **position 2** whenever the final token is $t_3$.

**Channel 0 becomes a position-specific content test.** Because the final query is concentrated on position 2, channel 0 of the value stream no longer asks *"is there a $t_1$ somewhere?"* It effectively asks *"is the token at position 2 equal to $t_1$?"*

$$
h_{\mathrm{last}}^{(0)} \approx v_2^{(0)} = \mathbf{1}\{t_{\mathrm{pos}\,2} = t_1\}
$$

**The output routing is unchanged.** Channel 1 is still the constant background value 1, so the same output matrix implements the two branches:

- if $h_{\mathrm{last}}^{(0)} \approx 1$, the $t_4$ bias is cancelled and $t_2$ wins
- if $h_{\mathrm{last}}^{(0)} \approx 0$, the default $t_4$ bias remains

This is the same algebra as Example 3, but with a different information source: the decision is now based on a **content-at-position** predicate rather than a **content-anywhere** predicate.

---

## Example 5: First-Token-Gated Position Rule

> Same as Example 4 in spirit (position-specific check at position 2), but the gate condition is now on the **first token**.

**Target rule:**

. *If the first token is $t_3$ and the token at position 2 is $t_1$, predict $t_2$. Otherwise predict $t_4$.*

### Design choices

- We keep the same key projection as Example 4: $W_K = e_{V+2}$, so only position 2 contributes a nonzero key.
- We still use two value channels and the same output routing idea as Example 3/4.
- The only structural change is that we read from the **first query row** (position 0) instead of the last row.

To make the first token act as a gate, we set query weights so token $t_3$ gives a positive query and all other tokens give a negative query:

$$
q = \begin{cases} +1 & \text{token is } t_3 \\ -1 & \text{token is not } t_3 \end{cases}
$$

Then for row 0:

- if token 0 is $t_3$, attention is pulled toward position 2
- otherwise, attention is pushed away from position 2

Channel 0 still detects token $t_1$ in the attended content; channel 1 remains the constant bias-like channel.

In [ ]:
alpha5 = 5
gamma5 = 5
d_v5 = 2

# QK setup:
# - key detects position 2 (same as Example 4)
# - query is +1 for token t3, -1 for all other tokens
W_Q5 = torch.nn.Linear(d, d_k, bias=False)
W_K5 = torch.nn.Linear(d, d_k, bias=False)

with torch.no_grad():
    W_Q5.weight.zero_()
    W_Q5.weight[0, :V] = -1.0
    W_Q5.weight[0, 3] = 1.0          # token t3 -> +1, all others -> -1
    W_K5.weight.zero_()
    W_K5.weight[0, V + 2] = 1.0      # key fires only at position 2

# Same two-channel value/output pattern as Example 4.
W_V5 = torch.zeros(d, d_v5)
W_V5[1, 0] = 1.0                     # channel 0: detects token t1
W_V5[:, 1] = 0.5 * torch.ones(d)     # channel 1: constant background = 1

W_O5 = torch.zeros(d_v5, V)
W_O5[0, 2] = gamma5
W_O5[0, 4] = -gamma5
W_O5[1, 4] = gamma5

def run_example5(seq, label):
    """Evaluate Example 5 and decode from the FIRST position row."""
    X_ = get_input_matrix(seq)

    Q_ = W_Q5(X_)
    K_ = W_K5(X_)
    S_ = alpha5 * (Q_ @ K_.T)
    P_ = F.softmax(S_, dim=-1)

    V_mat_ = X_ @ W_V5
    H_ = P_ @ V_mat_

    h_first_ = H_[0]                 # <-- decision row is position 0
    z_ = h_first_.unsqueeze(0) @ W_O5
    p_out_ = F.softmax(z_, dim=-1).squeeze()

    print(f"\n{'=' * 62}")
    print(f"Sequence: {seq}  [{label}]")
    print(f"  first-token query q0:      {Q_[0].item():.3f}")
    print(f"  K (pos-2 detector):        {K_.detach().numpy().flatten()}")
    print(f"  P[0] (first-row attention):{P_[0].detach().numpy().round(4)}")
    print(f"  V_mat (channel 0, 1):      {V_mat_.detach().numpy().T}")
    print(f"  h_first = [{h_first_[0].item():.4f}, {h_first_[1].item():.4f}]")
    print(f"  z (logits):                {z_.detach().numpy().round(4)}")
    print(f"  p_out: {[f't{i}={p:.3f}' for i, p in enumerate(p_out_.detach().numpy())]}")
    print(f"  -> Predicted: t{p_out_.argmax().item()}  (p={p_out_.max().item():.4f})")
    return P_.detach().numpy(), p_out_.detach().numpy(), h_first_.detach().numpy(), Q_.detach().numpy().flatten()

# True branch: first token is t3 and position 2 is t1 -> expect t2
P5a, pn5a, h5a, q5a = run_example5([3, 0, 1, 0], "first=t3 AND pos2=t1 -> expect t2")

# False branch A: first token is not t3, position 2 is t1 -> expect t4
P5b, pn5b, h5b, q5b = run_example5([0, 0, 1, 3], "first!=t3, pos2=t1 -> expect t4")

# False branch B: first token is t3, position 2 is not t1 -> expect t4
P5c, pn5c, h5c, q5c = run_example5([3, 0, 0, 1], "first=t3, pos2!=t1 -> expect t4")

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 12))

cases5 = [
    ([3, 0, 1, 0], "first=t3, pos2=t1 -> predict $t_2$", P5a, pn5a, h5a, 2),
    ([0, 0, 1, 3], "first!=t3, pos2=t1 -> predict $t_4$", P5b, pn5b, h5b, 4),
    ([3, 0, 0, 1], "first=t3, pos2!=t1 -> predict $t_4$", P5c, pn5c, h5c, 4),
]

for row, (seq, title, P_, pn, h_first, expected) in enumerate(cases5):
    pos_labels_ = [f"pos {i}\\n(t{seq[i]})" for i in range(L)]

    ax = axes[row, 0]
    im = ax.imshow(P_, cmap="Oranges", vmin=0, vmax=1)
    ax.set_title(f"$P$ — {title}", fontsize=10)
    ax.set_xlabel("Key pos")
    ax.set_ylabel("Query pos")
    ax.set_xticks(range(L)); ax.set_xticklabels(pos_labels_, fontsize=8)
    ax.set_yticks(range(L)); ax.set_yticklabels(pos_labels_, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
    for i in range(L):
        for j in range(L):
            ax.text(j, i, f"{P_[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if P_[i, j] > 0.5 else "black")

    ax = axes[row, 1]
    ch_colors = ["steelblue", "slategray"]
    bars = ax.bar(["ch 0\\n(token=$t_1$)", "ch 1\\n(constant)"], h_first, color=ch_colors, width=0.5)
    ax.set_ylim(0, 1.2)
    ax.set_title(r"$h_{\mathrm{first}}$ (2-channel output at pos 0)", fontsize=10)
    ax.set_ylabel("Value")
    for bar, val in zip(bars, h_first):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.03, f"{val:.3f}",
                ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.axhline(1.0, color="gray", lw=1, ls="--", alpha=0.5)

    ax = axes[row, 2]
    colors = ["steelblue"] * V
    colors[expected] = "darkorange"
    ax.bar(range(V), pn, color=colors)
    ax.set_ylim(0, 1.1)
    ax.set_title(r"$\hat{p}$ — predicted token", fontsize=10)
    ax.set_xlabel("Token")
    ax.set_ylabel("Probability")
    ax.set_xticks(range(V)); ax.set_xticklabels([f"t{i}" for i in range(V)])
    for i, p in enumerate(pn):
        ax.text(i, p + 0.02, f"{p:.3f}", ha="center", va="bottom", fontsize=9)

plt.suptitle(
    r"Example 5: first-token gate with $W_K=e_{V+2}$ and decode from row 0"
    f"\n$\\alpha={alpha5}$, $\\gamma={gamma5}$",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

## Interpretation: Example 5

**What changed from Example 4:** the gating condition is moved from the final token to the first token. To make that usable with a single head, we decode from the **first attention row** (position 0).

- If token 0 is $t_3$, the first-row query is positive and attention concentrates on position 2.
- If token 0 is not $t_3$, the first-row query is negative and attention avoids position 2.

Channel 0 reads token-$t_1$ content from the attended source, while channel 1 keeps the constant default bias. The same output matrix then implements the branch logic:

- strong channel-0 signal $\Rightarrow$ predict $t_2$
- weak channel-0 signal $\Rightarrow$ default to $t_4$

So this experiment is the first-token analogue of Example 4, with the same position-specific check at position 2 but a different gating location.

---

## Example 6: Same Rule as Example 5, But Decode from Final Row

Now we return to final-row decoding and keep a single attention head.

**Target rule:**

*If the first token is $t_3$ and the token at position 2 is $t_1$, predict $t_2$. Otherwise predict $t_4$.*

### Construction idea (single head, one layer)

- We force only the final row to carry signal by setting the query to a final-position indicator: $q_i = x_i[V+3]$.
- We keep a position-2 key detector, but also make position 0 a competing key whose strength depends on whether the first token is $t_3$.
- The final-row softmax then routes attention mostly to position 2 only in the true branch.

With one scalar head this is still soft logic, but with sufficiently large scale it becomes very sharp.

In [ ]:
alpha6 = 6
gamma6 = 5
d_v6 = 2

# Q: only final row is active (query scalar 1 at pos 3, else 0).
W_Q6 = torch.nn.Linear(d, d_k, bias=False)
W_K6 = torch.nn.Linear(d, d_k, bias=False)

with torch.no_grad():
    W_Q6.weight.zero_()
    W_Q6.weight[0, V + 3] = 1.0     # q_i = 1 only for final position i=3

    # Key design (single scalar key per position):
    # - Position 2 gets a moderate positive baseline; if token at pos2 is t1, gets extra boost.
    # - Position 0 competes strongly unless token0 is t3.
    # - Positions 1 and 3 are strongly suppressed.
    W_K6.weight.zero_()
    W_K6.weight[0, 3] = -4.0         # token t3 reduces key score (used at pos0)
    W_K6.weight[0, 1] = 4.0          # token t1 boosts key score (used at pos2)
    W_K6.weight[0, V + 0] = 4.0      # pos0 baseline
    W_K6.weight[0, V + 1] = -6.0     # suppress pos1
    W_K6.weight[0, V + 2] = -1.0     # pos2 baseline
    W_K6.weight[0, V + 3] = -6.0     # suppress pos3

# Values and output: same two-channel pattern as Example 5, but channel 0 is position-aware.
W_V6 = torch.zeros(d, d_v6)
W_V6[1, 0] = 1.0                     # +1 for token t1
W_V6[V + 0, 0] = -1.0               # -1 only at position 0 (prevents false positives there)
W_V6[:, 1] = 0.5 * torch.ones(d)     # constant channel => always 1 after attention

W_O6 = torch.zeros(d_v6, V)
W_O6[0, 2] = gamma6
W_O6[0, 4] = -gamma6
W_O6[1, 4] = gamma6

def run_example6(seq, label):
    """Single-head final-row decoder for the Example 6 rule."""
    X_ = get_input_matrix(seq)

    Q_ = W_Q6(X_)
    K_ = W_K6(X_)
    S_ = alpha6 * (Q_ @ K_.T)
    P_ = F.softmax(S_, dim=-1)

    V_mat_ = X_ @ W_V6
    H_ = P_ @ V_mat_

    h_last_ = H_[-1]
    z_ = h_last_.unsqueeze(0) @ W_O6
    p_out_ = F.softmax(z_, dim=-1).squeeze()

    print(f"\n{'=' * 64}")
    print(f"Sequence: {seq}  [{label}]")
    print(f"  Q (final-row selector):     {Q_.detach().numpy().flatten()}")
    print(f"  K (key scalars):            {K_.detach().numpy().flatten()}")
    print(f"  P[-1] (final-row attention):{P_[-1].detach().numpy().round(4)}")
    print(f"  V_mat (channel 0, 1):       {V_mat_.detach().numpy().T}")
    print(f"  h_last = [{h_last_[0].item():.4f}, {h_last_[1].item():.4f}]")
    print(f"  z (logits):                 {z_.detach().numpy().round(4)}")
    print(f"  p_out: {[f't{i}={p:.3f}' for i, p in enumerate(p_out_.detach().numpy())]}")
    print(f"  -> Predicted: t{p_out_.argmax().item()}  (p={p_out_.max().item():.4f})")
    return P_.detach().numpy(), p_out_.detach().numpy(), h_last_.detach().numpy(), K_.detach().numpy().flatten()

# True branch: first=t3 and pos2=t1 -> expect t2
P6a, pn6a, h6a, k6a = run_example6([3, 0, 1, 0], "first=t3 AND pos2=t1 -> expect t2")

# False branch A: first!=t3 and pos2=t1 -> expect t4
P6b, pn6b, h6b, k6b = run_example6([0, 0, 1, 3], "first!=t3, pos2=t1 -> expect t4")

# False branch B: first=t3 and pos2!=t1 -> expect t4
P6c, pn6c, h6c, k6c = run_example6([3, 0, 0, 1], "first=t3, pos2!=t1 -> expect t4")

# False branch C: both false, with token t1 at position 0 to stress-test robustness
P6d, pn6d, h6d, k6d = run_example6([1, 2, 0, 4], "first!=t3, pos2!=t1 -> expect t4")

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 15))

cases6 = [
    ([3, 0, 1, 0], "first=t3, pos2=t1 -> predict $t_2$", P6a, pn6a, h6a, 2),
    ([0, 0, 1, 3], "first!=t3, pos2=t1 -> predict $t_4$", P6b, pn6b, h6b, 4),
    ([3, 0, 0, 1], "first=t3, pos2!=t1 -> predict $t_4$", P6c, pn6c, h6c, 4),
    ([1, 2, 0, 4], "both false -> predict $t_4$", P6d, pn6d, h6d, 4),
]

for row, (seq, title, P_, pn, h_last, expected) in enumerate(cases6):
    pos_labels_ = [f"pos {i}\\n(t{seq[i]})" for i in range(L)]

    ax = axes[row, 0]
    im = ax.imshow(P_, cmap="Oranges", vmin=0, vmax=1)
    ax.set_title(f"$P$ — {title}", fontsize=10)
    ax.set_xlabel("Key pos")
    ax.set_ylabel("Query pos")
    ax.set_xticks(range(L)); ax.set_xticklabels(pos_labels_, fontsize=8)
    ax.set_yticks(range(L)); ax.set_yticklabels(pos_labels_, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
    for i in range(L):
        for j in range(L):
            ax.text(j, i, f"{P_[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if P_[i, j] > 0.5 else "black")

    ax = axes[row, 1]
    ch_colors = ["steelblue", "slategray"]
    bars = ax.bar(["ch 0\\n(position-aware $t_1$)", "ch 1\\n(constant)"], h_last, color=ch_colors, width=0.5)
    ax.set_ylim(-1.2, 1.2)
    ax.set_title(r"$h_{\mathrm{last}}$ (final-row 2-channel output)", fontsize=10)
    ax.set_ylabel("Value")
    for bar, val in zip(bars, h_last):
        ax.text(bar.get_x() + bar.get_width() / 2, val + (0.04 if val >= 0 else -0.08), f"{val:.3f}",
                ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.axhline(0.0, color="gray", lw=1, ls="--", alpha=0.5)
    ax.axhline(1.0, color="gray", lw=1, ls="--", alpha=0.3)

    ax = axes[row, 2]
    colors = ["steelblue"] * V
    colors[expected] = "darkorange"
    ax.bar(range(V), pn, color=colors)
    ax.set_ylim(0, 1.1)
    ax.set_title(r"$\hat{p}$ — predicted token", fontsize=10)
    ax.set_xlabel("Token")
    ax.set_ylabel("Probability")
    ax.set_xticks(range(V)); ax.set_xticklabels([f"t{i}" for i in range(V)])
    for i, p in enumerate(pn):
        ax.text(i, p + 0.02, f"{p:.3f}", ha="center", va="bottom", fontsize=9)

plt.suptitle(
    r"Example 6: single-head final-row decoding for first-token AND position-2 rule"
    f"\n$\\alpha={alpha6}$, $\\gamma={gamma6}$",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

## Interpretation: Example 6

This construction keeps **single-head, single-layer attention** and still decodes from the final row.

The crucial mechanism is competition in the final-row softmax:

- Position 2 gets high key score when its token is $t_1$.
- Position 0 gets high key score unless the first token is $t_3$.

So the final row attends to position 2 strongly only in the true branch $(\text{first}=t_3) \land (\text{pos2}=t_1)$.

Channel 0 is position-aware (`+t1` and `-pos0`), which prevents accidental activation from token $t_1$ at position 0. Channel 1 provides the default bias-like path to $t_4$.

Result: final-row decoding can implement this rule with one head in this toy setup.

## Comparison: Examples 4 vs 5 vs 6

| Item | Example 4 | Example 5 | Example 6 |
|---|---|---|---|
| Rule intent | first fixed to $t_3$, check pos2=$t_1$ | first=$t_3$ and pos2=$t_1$ | first=$t_3$ and pos2=$t_1$ |
| Decode row | final row ($L-1$) | **first row (0)** | **final row ($L-1$)** |
| How the first-token condition is used | implicit in setup (fixed-final-token framing) | direct query gate at row 0 | encoded via key competition seen by final-row softmax |
| Position-2 check | $W_K=e_{V+2}$ | $W_K=e_{V+2}$ | $W_K$ includes pos2 boost and competing pos0 term |
| Value channels | 2 channels (token detector + constant) | 2 channels (token detector + constant) | 2 channels (position-aware token detector + constant) |
| Single head / single layer | yes | yes | yes |
| Best takeaway | position-specific readout | first-token gate is easy if decoding from row 0 | first-token gate can still be done while decoding from final row |

In short: Example 5 demonstrates the concept most directly, while Example 6 shows the same logical rule can be realized with strict final-row decoding in a single-head, single-layer toy construction.

In [ ]:
W_Q = W_Q6.weight.transpose(0,1)
W_K = W_K6.weight.transpose(0,1)
X = get_input_matrix([3, 0, 1, 0])
Q = W_Q6(X)
K = W_K6(X)
print("W_Q (d_k × d):")
print(W_Q.detach().numpy())
print("\nW_K (d_k × d):")
print(W_K.detach().numpy())
print("\nInput matrix X (L × d):")
print(X.detach().numpy())
print("\nQuery matrix Q = X @ W_Q (L × d_k):")
print(Q.detach().numpy())
print("\nKey matrix K = X @ W_K (L × d_k):")
print(K.detach().numpy())

In [ ]:
S = alpha6 * (Q @ K.T)
P = F.softmax(S, dim=-1)
print("\nAttention scores S = alpha * (Q @ K^T) (L × L):")
print(S.detach().numpy())
print("\nAttention weights P (L × L):")
print(P.detach().numpy())

In [ ]:
W_V = W_V6
W_O = W_O6
V_mat = X @ W_V
H = P @ V_mat

h_last = H[-1]
z = h_last.unsqueeze(0) @ W_O
p_out = F.softmax(z, dim=-1).squeeze()

print("\nValue matrix W_V (d × d_v):")
print(W_V.detach().numpy())
print("\nOutput matrix W_O (d_v × V):")
print(W_O.detach().numpy())
print("\nValue matrix V_mat = X @ W_V (L × d_v):")
print(V_mat.detach().numpy())
print("\nAttention output H = P @ V_mat (L × d_v):")
print(H.detach().numpy())
print("\nFinal attention output h_last (d_v):")
print(h_last.detach().numpy())
print("\nLogits z = h_last @ W_O (1 × V):")
print(z.detach().numpy())

print("\nOutput probabilities p_out:")
print(p_out.detach().numpy())